In [1]:
import pandas as pd
import glob
import os
import re
import random

# Functions

In [2]:
def process_files(files):

    dfs = []
    for file in files:
        # Leer el archivo csv
        df = pd.read_csv(file)

        #print("file: ", file)
        # 3. Extraer 'alt' y 'raan' del nombre del archivo usando expresiones regulares
        alt_match = re.search(r'alt_(\d+)', file)
        raan_match = re.search(r'raan_(\d+)', file)

        #df['alt'] = alt_match
        #df['raan'] = raan_match


        if alt_match:
            df['alt'] = int(alt_match.group(1))
        if raan_match:
            df['raan'] = int(raan_match.group(1))

        df['id_simulation'] = 'sim_alt_' +str(int(alt_match.group(1)))+ '_raan_' + str(int(raan_match.group(1)))

        ''' 
        print(df.shape)
        print('id_simulation: ', 'sim_alt_' +str(int(alt_match.group(1)))+ '_raan_' + str(int(raan_match.group(1))))
        print('0 : ', df[df['rcvOk'] == 0].shape[0], ' | ' )#,round(df[df['rcvOk'] == 0].shape[0] * 100 / df.shape[0],2))
        print('1 : ', df[df['rcvOk'] == 1].shape[0], ' | ' )#,round(df[df['rcvOk'] == 1].shape[0] * 100 / df.shape[0],2))
        '''
        dfs.append(df)

    # 4. Concatenar todos los DataFrames
    if dfs:
        final_df = pd.concat(dfs, ignore_index=True)

        # Guardar el resultado final en un nuevo archivo
        #output_filename = "Concatenated_TransmissionStatistics.csv"
        #final_df.to_csv(output_filename, index=False)

        #print(f"Se concatenaron {len(files)} archivos con éxito.")
        #print(final_df.head())
    else:
        print("No se encontraron archivos que coincidan con el patrón.")

    
    #final_df.to_csv('final_data_tranmissions.csv', index=False)
    return final_df

# Seed

In [3]:
base_path = os.path.join("results", "repetitions")

# Getting seed folders
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))
seed_folders

['results/repetitions/seed_0/',
 'results/repetitions/seed_1/',
 'results/repetitions/seed_2/',
 'results/repetitions/seed_3/',
 'results/repetitions/seed_4/',
 'results/repetitions/seed_5/',
 'results/repetitions/seed_6/',
 'results/repetitions/seed_7/',
 'results/repetitions/seed_8/',
 'results/repetitions/seed_9/']

In [4]:
dict_train_files = {}
dict_test_files = {}


dict_test_files[600] = [200]
dict_test_files[850] = [330]
dict_test_files[740] = [150, 180]
dict_test_files[700] = [190]
dict_test_files

{600: [200], 850: [330], 740: [150, 180], 700: [190]}

In [7]:
# Ruta base donde están las carpetas seed_X
base_path = os.path.join("results", "repetitions")

# Getting seed folders
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))

# Diccionarios para organizar los resultados por seed si lo necesitas
# O simplemente listas globales si quieres mezclar todas las semillas al final
all_train_pathfiles = []
all_test_pathfiles = []

# 1. Obtener todas las carpetas que coincidan con 'seed_*'
seed_folders = sorted(glob.glob(os.path.join(base_path, "seed_*/")))
print('seed_folders: ', seed_folders)
#seed_folders

for seed_folder in seed_folders:
    print(f"--- Procesando: {(seed_folder)} ---")
    
    # 2. Identificar archivos dentro de la carpeta seed actual
    file_pattern = os.path.join(seed_folder, "TransmissionStatistics_*.csv")
    files_in_seed = glob.glob(file_pattern)

    print('files_in_seed: ', len(files_in_seed))
    train_pathfiles = []
    test_pathfiles  = []

    alt_identified = []
    for file in files_in_seed:
        alt_match = re.search(r'alt_(\d+)', file)
        raan_match = re.search(r'raan_(\d+)', file)
        alt_identified.append(int(alt_match.group(1)))

    alt_identified = set(alt_identified)
    print(alt_identified)

    new_file_pattern = os.path.join(seed_folder, "TransmissionStatistics_alt*.csv")

    for alt in alt_identified:
        print(">>alt: ", alt)
        aux = "TransmissionStatistics_alt_" + str(alt) + "*.csv"
        new_file_pattern = os.path.join(seed_folder, aux)
        new_files = glob.glob(new_file_pattern)
        
        ###num_files_train = int(len(new_files)*0.8)
        ###num_files_test  = len(new_files) - num_files_train        
        
        ###train_files_selected = random.sample(new_files, num_files_train)
        ###test_files_selected = [f for f in new_files if f not in train_files_selected]

        train_files_selected = []
        test_files_selected = []

        for file in new_files:
            #alt_match = re.search(r'alt_(\d+)', file)
            raan_match = re.search(r'raan_(\d+)', file)
            raan_identified = int(raan_match.group(1))
            #print('raan_identified: ',raan_identified)
            if alt in dict_test_files.keys():
                if raan_identified in dict_test_files[alt]:
                    test_files_selected.append(file)
                else:
                    train_files_selected.append(file)

        train_pathfiles.extend(train_files_selected)
        test_pathfiles.extend(test_files_selected)
        
    print(">test_files_selected: ", len(test_pathfiles), test_pathfiles)
    print(">train_pathfiles: ", len(train_pathfiles), train_pathfiles)

    #''' 
    train_csv = process_files(train_pathfiles)
    test_csv  = process_files(test_pathfiles)

    all_test_pathfiles.extend(test_pathfiles)
    all_train_pathfiles.extend(train_pathfiles)

    train_csv.to_csv(seed_folder + '_final_train_data_transmission.csv', index=False)

    # 1. Aseguramos el orden
    df_sorted = test_csv.sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

    # 2. Calculamos el punto de corte
    split_point = len(df_sorted) // 2

    # 3. Dividimos manteniendo el orden
    test_df = df_sorted.iloc[:split_point].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)
    val_df = df_sorted.iloc[split_point:].sort_values(by=['id_simulation', 'time']).reset_index(drop=True)

    test_df.to_csv(seed_folder + '_final_test_data_transmission.csv', index=False)
    val_df.to_csv(seed_folder + '_final_val_data_transmission.csv', index=False)

    print(f"Total: {len(df_sorted)} | Test: {len(test_df)} | Val: {len(val_df)}")
    #'''

seed_folders:  ['results/repetitions/seed_0/', 'results/repetitions/seed_1/', 'results/repetitions/seed_2/', 'results/repetitions/seed_3/', 'results/repetitions/seed_4/', 'results/repetitions/seed_5/', 'results/repetitions/seed_6/', 'results/repetitions/seed_7/', 'results/repetitions/seed_8/', 'results/repetitions/seed_9/']
--- Procesando: results/repetitions/seed_0/ ---
files_in_seed:  21
{600, 850, 700, 740}
>>alt:  600
>>alt:  850
>>alt:  700
>>alt:  740
>test_files_selected:  5 ['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_200_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_850_raan_330_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_700_raan_190_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_150_seed_0.csv', 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_180_seed_0.csv']
>train_pathfiles:  16 ['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_170_seed_0.csv', 're

In [6]:
test_pathfiles

['results/repetitions/seed_9/TransmissionStatistics_alt_600_raan_200_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_850_raan_330_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_700_raan_190_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_740_raan_180_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_740_raan_150_seed_9.csv']

In [8]:
all_test_pathfiles

['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_200_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_850_raan_330_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_700_raan_190_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_150_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_180_seed_0.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_600_raan_200_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_700_raan_190_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_740_raan_180_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_740_raan_150_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_850_raan_330_seed_1.csv',
 'results/repetitions/seed_2/TransmissionStatistics_alt_600_raan_200_seed_2.csv',
 'results/repetitions/seed_2/TransmissionStatistics_alt_850_raan_330_seed_2.csv',
 'results/repeti

In [ ]:
['results/repetitions/seed_0/TransmissionStatistics_alt_600_raan_200_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_700_raan_190_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_150_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_740_raan_180_seed_0.csv',
 'results/repetitions/seed_0/TransmissionStatistics_alt_850_raan_330_seed_0.csv', 


 'results/repetitions/seed_1/TransmissionStatistics_alt_600_raan_200_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_700_raan_190_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_740_raan_180_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_740_raan_150_seed_1.csv',
 'results/repetitions/seed_1/TransmissionStatistics_alt_850_raan_330_seed_1.csv',


 'results/repetitions/seed_2/TransmissionStatistics_alt_600_raan_200_seed_2.csv',
 'results/repetitions/seed_2/TransmissionStatistics_alt_700_raan_190_seed_2.csv',
 'results/repetitions/seed_2/TransmissionStatistics_alt_740_raan_150_seed_2.csv',
 'results/repetitions/seed_2/TransmissionStatistics_alt_740_raan_180_seed_2.csv',
 'results/repetitions/seed_2/TransmissionStatistics_alt_850_raan_330_seed_2.csv',


 'results/repetitions/seed_3/TransmissionStatistics_alt_600_raan_200_seed_3.csv',
 'results/repetitions/seed_3/TransmissionStatistics_alt_700_raan_190_seed_3.csv',
 'results/repetitions/seed_3/TransmissionStatistics_alt_740_raan_180_seed_3.csv',
 'results/repetitions/seed_3/TransmissionStatistics_alt_740_raan_150_seed_3.csv',
 'results/repetitions/seed_3/TransmissionStatistics_alt_850_raan_330_seed_3.csv',


 'results/repetitions/seed_4/TransmissionStatistics_alt_600_raan_200_seed_4.csv',
 'results/repetitions/seed_4/TransmissionStatistics_alt_700_raan_190_seed_4.csv',
 'results/repetitions/seed_4/TransmissionStatistics_alt_740_raan_180_seed_4.csv',
 'results/repetitions/seed_4/TransmissionStatistics_alt_740_raan_150_seed_4.csv',
 'results/repetitions/seed_4/TransmissionStatistics_alt_850_raan_330_seed_4.csv',


 'results/repetitions/seed_5/TransmissionStatistics_alt_600_raan_200_seed_5.csv',
 'results/repetitions/seed_5/TransmissionStatistics_alt_700_raan_190_seed_5.csv',
 'results/repetitions/seed_5/TransmissionStatistics_alt_740_raan_150_seed_5.csv',
 'results/repetitions/seed_5/TransmissionStatistics_alt_740_raan_180_seed_5.csv',
 'results/repetitions/seed_5/TransmissionStatistics_alt_850_raan_330_seed_5.csv',


 'results/repetitions/seed_6/TransmissionStatistics_alt_600_raan_200_seed_6.csv',
 'results/repetitions/seed_6/TransmissionStatistics_alt_700_raan_190_seed_6.csv',
 'results/repetitions/seed_6/TransmissionStatistics_alt_740_raan_180_seed_6.csv',
 'results/repetitions/seed_6/TransmissionStatistics_alt_740_raan_150_seed_6.csv',
 'results/repetitions/seed_6/TransmissionStatistics_alt_850_raan_330_seed_6.csv',


 'results/repetitions/seed_7/TransmissionStatistics_alt_600_raan_200_seed_7.csv',
 'results/repetitions/seed_7/TransmissionStatistics_alt_700_raan_190_seed_7.csv',
 'results/repetitions/seed_7/TransmissionStatistics_alt_740_raan_150_seed_7.csv',
 'results/repetitions/seed_7/TransmissionStatistics_alt_740_raan_180_seed_7.csv',
 'results/repetitions/seed_7/TransmissionStatistics_alt_850_raan_330_seed_7.csv',


 'results/repetitions/seed_8/TransmissionStatistics_alt_600_raan_200_seed_8.csv',
 'results/repetitions/seed_8/TransmissionStatistics_alt_700_raan_190_seed_8.csv',
 'results/repetitions/seed_8/TransmissionStatistics_alt_740_raan_180_seed_8.csv',
 'results/repetitions/seed_8/TransmissionStatistics_alt_740_raan_150_seed_8.csv',
 'results/repetitions/seed_8/TransmissionStatistics_alt_850_raan_330_seed_8.csv',


 'results/repetitions/seed_9/TransmissionStatistics_alt_600_raan_200_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_700_raan_190_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_740_raan_180_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_740_raan_150_seed_9.csv',
 'results/repetitions/seed_9/TransmissionStatistics_alt_850_raan_330_seed_9.csv',]